In [ ]:
#---------------------------------REFORMULATING AGENT----------------------------------------
import os
from openai import OpenAI

# Set your key (use %env OPENROUTER_API_KEY=sk-or-... in a cell for safety)
os.environ["OPENROUTER_API_KEY"] = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
      default_headers={
        "HTTP-Referer": "https://colab.research.google.com/drive/12NI_KC7ESexhwJzQ8_8kSDSKwpnHT_5n?authuser=1#scrollTo=Y3pz5QeOxm0j",  # Your notebook/app URL (required for local)
        "X-Title": "Query Reformulator",             # Your app/project name
    }
)
SYSTEM_PROMPT = (
    "You are a query rewriting assistant. "
    "Given a user query, you rewrite it into a clearer, more complete search query. "
    "Preserve the original intent. "
    "Return only the rewritten query, with no explanations."
)

def reformulate_query(query: str) -> str:
    response = client.chat.completions.create(
        model="deepseek/deepseek-v3.2",  # for example
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ],
        temperature=0.2,
        max_tokens=64,
    )
    content = response.choices[0].message.content.strip()
    if not content:
      raise RuntimeError("No choices returned")  
    return content

# Example usage
if __name__ == "__main__":
    original = "best way to speed up my postgres queries?"
    new_query = reformulate_query(original)
    print("Original:", original)
    print("Rewritten:", new_query)


In [ ]:
#-----------------------------------RE-RANKING-AGENT------------------------------------------------
from sentence_transformers import CrossEncoder
from typing import List, Dict, Any

class RerankingAgent:
    """
    A re-ranking agent that uses a cross-encoder to re-score the top 50 documents.
    Assumes input is top 50 documents from a retriever.
    """
    def __init__(self, model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'):
        """
        Initialize the reranking agent with the specified cross-encoder model.

        Args:
            model_name: Hugging Face model name for the cross-encoder [web:16][page:0].
        """
        self.model = CrossEncoder(model_name)  # Small, fast model ~22M params [web:16].

    def rerank(self, query: str, documents: List[Dict[str, Any]], top_k: int = 10) -> List[Dict[str, Any]]:
        """
        Re-rank the top 50 documents using the cross-encoder.

        Args:
            query: The search query string.
            documents: List of top 50 documents, each as dict with 'id', 'text', etc.
            top_k: Number of top re-ranked results to return.

        Returns:
            Re-ranked list of documents with added 'rerank_score' field [web:6][web:19].
        """
        # Create query-document pairs for batch scoring
        pairs = [[query, doc['text']] for doc in documents]  # Efficient batching [web:6].

        # Compute relevance scores (higher = more relevant)
        scores = self.model.predict(pairs)  # Scores between 0-1 typically [page:0].

        # Add scores and sort descending
        for doc, score in zip(documents, scores):
            doc['rerank_score'] = float(score)

        documents.sort(key=lambda x: x['rerank_score'], reverse=True)
        return documents[:top_k]

# Usage example:
# agent = RerankingAgent()
# top_50_docs = [...]  # From retriever, e.g., top 50 vector search results
# reranked = agent.rerank("What is Python?", top_50_docs, top_k=5)

In [ ]:
# ---- Test code ----
def test_reranking_agent():
    # Example query
    query = "How many people live in Berlin?"

    # Mock 'top 50' documents (here just a few for brevity)
    documents = [
        {
            "id": 1,
            "text": "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."
        },
        {
            "id": 2,
            "text": "Berlin is well known for its museums."
        },
        {
            "id": 3,
            "text": "The urban area of Berlin comprised about 4.1 million people in 2014, making it the seventh most populous urban area in the European Union."
        },
        {
            "id": 4,
            "text": "The city of Paris had a population of 2,165,423 people within its administrative city limits as of January 1, 2019."
        },
    ]
    # In a real test, you would provide up to 50 retrieved documents here [web:22][web:38].

    agent = RerankingAgent()
    reranked = agent.rerank(query, documents, top_k=3)

    print("Query:", query)
    print("Top reranked results:")
    for doc in reranked:
        print(f"- id={doc['id']}, score={doc['rerank_score']:.4f}, text={doc['text']}")

    # Simple sanity checks
    assert len(reranked) == 3
    # Check that scores are sorted descending
    scores = [d["rerank_score"] for d in reranked]
    assert scores == sorted(scores, reverse=True)
    print("Sanity checks passed.")


if __name__ == "__main__":
    test_reranking_agent()

In [ ]:
Query = str
DocId = str
class OrcasPriorLookup:
    def __init__(self, prior_map: Dict[Tuple[Query, DocId], float]):
        self.prior_map = prior_map

    def get_prior(self, query_text: str, doc_id: str) -> float:
        """
        Return prior for (query_text, doc_id), or 0.0 if unseen.
        """
        return self.prior_map.get((query_text, doc_id), 0.0)

In [ ]:
from typing import List, Dict, Any

class ClickPriorAgent:
    def __init__(self, beta: float, prior_lookup: OrcasPriorLookup):
        """
        beta: scalar weight β in score' = score + β * prior.
        prior_lookup: wrapper around ORCAS prior map.
        """
        self.beta = beta
        self.prior_lookup = prior_lookup

    def reweight_results(
        self,
        query_text: str,
        hits: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """
        hits: list of docs, each with at least:
          {
            "doc_id": str,   # must match ORCAS did field
            "score": float,  # original BM25 / model score
            ...              # any extra fields are kept
          }
        Returns a new list with updated scores and re-sorted by new_score.
        """
        reweighted = []
        for h in hits:
            #doc_id = h["doc_id"]
            doc_id = h["_source"]["docid"]
            base_score = float(h["_score"])
            prior = self.prior_lookup.get_prior(query_text, doc_id)
            new_score = base_score + self.beta * prior

            h_new = dict(h)
            h_new["prior"] = prior
            h_new["score_before_prior"] = base_score
            h_new["score"] = new_score  # overwrite with reweighted score
            reweighted.append(h_new)

        # Sort descending by new score
        reweighted.sort(key=lambda x: x["score"], reverse=True)
        return reweighted

In [ ]:

import requests
import json

# 1) Load priors once at startup
prior_map = load_orcas_priors("/content/orcas.tsv")
prior_lookup = OrcasPriorLookup(prior_map)
agent = ClickPriorAgent(beta=0.1, prior_lookup=prior_lookup)

# 2) For each user query:
query_text = "Restaurants in Passau"

# hits = ... transform from OpenSearch response to list of dicts
# hits = [{"doc_id": "D123", "score": 12.3, "_source": {...}}, ...]

url = "https://opensearch.pads.fim.uni-passau.de/msmarco-v2.1-segmented/_search"
auth = ("hana", "4DBpreroRMc!fPPczxaJ")
query = "Restaurants in Passau"

params = {
    "query": {"match": {"segment": query}},
    "size": 50,  # top-50
    "_source": ["docid", "segment"]  # Extract docid + text
}

resp = requests.get(url, auth=auth, json=params)
hits = resp.json()["hits"]["hits"]
reweighted_hits = agent.reweight_results(query_text, hits)

# reweighted_hits now contains:
# - updated "score" = BM25 + β * prior
# - added "prior" and "score_before_prior"
